## Catboost

In [ ]:
print("\n" + "="*60)
print("3. CatBoost ON ORIGINAL FEATURES (no preprocessing)")
print("="*60)

# Use raw data (string categories, non-normalized numbers)
X_cb = X_ft_raw.copy()
X_test_cb = X_test_raw.copy()
y_cb = y_ft_raw.copy()
y_test_cb = y_test_raw.copy()

# Convert categorical columns to strings (required by CatBoost)
for col in cat_cols:
    X_cb[col] = X_cb[col].astype(str)
    X_test_cb[col] = X_test_cb[col].astype(str)

print(f"Data size: {X_cb.shape}")
print(f"Categorical features: {len(cat_cols)}")
print(f"Categorical column names: {cat_cols}")

# ============================================================
# GRID SEARCH FOR CATBOOST
# ============================================================
# Define hyperparameter grid for CatBoost
param_grid_cb = {
    'learning_rate': [0.05, 0.1],
    'depth': [4, 6, 8],
    'l2_leaf_reg': [5]
}
all_params = list(ParameterGrid(param_grid_cb))

best_score = -1
best_params = {}
best_iterations = 100
print(f"Total combinations: {len(all_params)}")
print(f"Cross-validation: 5 folds")
print("-" * 60)

# Perform grid search with cross-validation
for params in tqdm(all_params, desc='CatBoost Grid Search'):
    prauc_scores = []
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    fold_best_iters = []

    for train_idx, val_idx in cv.split(X_cb, y_cb):
        X_train, X_val = X_cb.iloc[train_idx], X_cb.iloc[val_idx]
        y_train, y_val = y_cb.iloc[train_idx], y_cb.iloc[val_idx]

        # Train CatBoost model with early stopping
        cb_model = CatBoostClassifier(
            **params,
            cat_features=cat_cols,          # Column names (not indices)
            auto_class_weights='Balanced',  # Handle class imbalance
            iterations=1000,
            early_stopping_rounds=50,
            verbose=0,
            random_state=seed,
            eval_metric='PRAUC'  # Use PR-AUC as evaluation metric
        )
        cb_model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True, verbose=0)

        fold_best_iters.append(cb_model.get_best_iteration())
        preds = cb_model.predict_proba(X_val)[:, 1]
        prauc = average_precision_score(y_val, preds)
        prauc_scores.append(prauc)

    mean_prauc = np.mean(prauc_scores)
    mean_best_iter = int(np.mean(fold_best_iters))

    # Update best parameters if improvement found
    if mean_prauc > best_score:
        best_score = mean_prauc
        best_params = params
        best_iterations = mean_best_iter
        print(f"\n✓ New best! PRAUC (CV): {best_score:.4f} | Best iterations: {best_iterations} | Params: {best_params}")

print("\n" + "="*60)
print("CATBOOST GRID SEARCH RESULTS")
print("="*60)
print(f"Best PRAUC (CV): {best_score:.4f}")
print(f"Best parameters: {best_params}")
print(f"Optimal iterations: {best_iterations}")
print("="*60)

# ============================================================
# TRAIN FINAL CATBOOST MODEL
# ============================================================
# Train final model on full finetune data with best parameters
best_cb_model = CatBoostClassifier(
    **best_params,
    cat_features=cat_cols,
    auto_class_weights='Balanced',
    iterations=best_iterations,
    verbose=50,  # Show training progress
    random_state=seed,
    eval_metric='PRAUC'
)
best_cb_model.fit(X_cb, y_cb, verbose=50)

# Evaluate on test set
y_pred_cb = best_cb_model.predict(X_test_cb)
probs_cb = best_cb_model.predict_proba(X_test_cb)[:, 1]

print("\n--- Test set ---")
print(f"AUPRC: {average_precision_score(y_test_cb, probs_cb):.4f}")
print(f"F1: {f1_score(y_test_cb, y_pred_cb):.4f}")
print(f"Recall: {recall_score(y_test_cb, y_pred_cb):.4f}")
print(classification_report(y_test_cb, y_pred_cb))